In [1]:
import pandas as pd
from dateutil import parser
import pytz
import difflib
from rapidfuzz import fuzz

/Users/will/Applications/miniconda3/lib/python3.11/site-packages/pandas/core/arrays/masked.py:61: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


In [2]:
cfbd = pd.read_csv("/Users/will/GitHub/IS477/data/raw/cfbd_merged.csv", low_memory = False)
box = pd.read_csv("/Users/will/GitHub/IS477/data/raw/cfb_box-scores_2002-2024.csv", low_memory = False)

In [3]:
cfbd.columns

Index(['id_x', 'season', 'week', 'seasonType', 'startDate', 'startTimeTBD',
       'completed', 'neutralSite', 'conferenceGame', 'attendance', 'venueId',
       'venue', 'homeId', 'homeTeam', 'homeClassification', 'homeConference',
       'homePoints', 'homeLineScores', 'homePostgameWinProbability',
       'homePregameElo', 'homePostgameElo', 'awayId', 'awayTeam',
       'awayClassification', 'awayConference', 'awayPoints', 'awayLineScores',
       'awayPostgameWinProbability', 'awayPregameElo', 'awayPostgameElo',
       'excitementIndex', 'highlights', 'notes', 'id_y', 'name', 'capacity',
       'grass', 'dome', 'city', 'state', 'zip', 'countryCode', 'timezone',
       'latitude', 'longitude', 'elevation', 'constructionYear'],
      dtype='object')

In [4]:
print(len(cfbd))
print(len(box))

42294
18909


In [5]:
box.columns

Index(['season', 'week', 'date', 'time_et', 'game_type', 'away', 'home',
       'rank_away', 'rank_home', 'conf_away', 'conf_home', 'neutral',
       'score_away', 'score_home', 'q1_away', 'q2_away', 'q3_away', 'q4_away',
       'ot_away', 'q1_home', 'q2_home', 'q3_home', 'q4_home', 'ot_home',
       'first_downs_away', 'first_downs_home', 'third_down_comp_away',
       'third_down_att_away', 'third_down_comp_home', 'third_down_att_home',
       'fourth_down_comp_away', 'fourth_down_att_away',
       'fourth_down_comp_home', 'fourth_down_att_home', 'pass_comp_away',
       'pass_att_away', 'pass_yards_away', 'pass_comp_home', 'pass_att_home',
       'pass_yards_home', 'rush_att_away', 'rush_yards_away', 'rush_att_home',
       'rush_yards_home', 'total_yards_away', 'total_yards_home', 'fum_away',
       'fum_home', 'int_away', 'int_home', 'pen_num_away', 'pen_yards_away',
       'pen_num_home', 'pen_yards_home', 'possession_away', 'possession_home',
       'attendance', 'tv'],
      dt

In [10]:
# -----------------------------------------------------------
# Prep functions
# -----------------------------------------------------------

def prepare_sched(sched: pd.DataFrame) -> pd.DataFrame:
    """
    Prepare the schedule dataframe (this is your `box` df).

    Expected columns:
      - season, week
      - date (e.g. '2002-08-29')
      - time_et (e.g. '7:00 PM')
      - home, away
      - attendance (optional)
    """
    sched = sched.copy()
    sched = (
        sched.reset_index(drop=True)
             .reset_index()
             .rename(columns={'index': 'sched_game_id'})
    )

    # Local ET datetime from date + time_et
    sched['game_dt_et'] = pd.to_datetime(
        sched['date'].astype(str) + ' ' + sched['time_et'].astype(str),
        errors='coerce'
    )

    sched['date_key'] = sched['game_dt_et'].dt.date
    sched['time_key'] = sched['game_dt_et'].dt.round('5min')

    # Make sure season/week are numeric, but allow NaNs (no int casting!)
    sched['season'] = pd.to_numeric(sched['season'], errors='coerce')
    sched['week'] = pd.to_numeric(sched['week'], errors='coerce')

    # Attendance numeric, allow NaNs, keep as float
    if 'attendance' in sched.columns:
        sched['attendance'] = pd.to_numeric(
            sched['attendance'], errors='coerce'
        ).round()
    else:
        sched['attendance'] = pd.NA

    return sched


def prepare_cfbd_box(cfbd: pd.DataFrame) -> pd.DataFrame:
    """
    Prepare the cfbd games/box dataframe (this is your `cfbd` df).

    Expected columns:
      - season, week
      - startDate (UTC ISO string, e.g. '2002-08-29T23:00:00.000Z')
      - homeTeam, awayTeam
      - attendance (optional)
    """
    cfbd = cfbd.copy()
    cfbd = (
        cfbd.reset_index(drop=True)
            .reset_index()
            .rename(columns={'index': 'cfbd_game_id'})
    )

    # startDate is UTC timestamp
    cfbd['game_dt_utc'] = pd.to_datetime(
        cfbd['startDate'], utc=True, errors='coerce'
    )
    cfbd['game_dt_et'] = (
        cfbd['game_dt_utc']
            .dt.tz_convert('US/Eastern')
            .dt.tz_localize(None)
    )

    cfbd['date_key'] = cfbd['game_dt_et'].dt.date
    cfbd['time_key'] = cfbd['game_dt_et'].dt.round('5min')

    cfbd['season'] = pd.to_numeric(cfbd['season'], errors='coerce')
    cfbd['week'] = pd.to_numeric(cfbd['week'], errors='coerce')

    if 'attendance' in cfbd.columns:
        cfbd['attendance'] = pd.to_numeric(
            cfbd['attendance'], errors='coerce'
        ).round()
    else:
        cfbd['attendance'] = pd.NA

    return cfbd


# -----------------------------------------------------------
# Matching helpers
# -----------------------------------------------------------

def add_team_match_score(candidates: pd.DataFrame) -> pd.DataFrame:
    """
    Add a fuzzy match score based on home/away team names.

    Requires columns:
      - home_sched, away_sched
      - home_cfbd, away_cfbd
    """
    def _calc(row):
        home_sched = str(row['home_sched'])
        away_sched = str(row['away_sched'])
        home_cfbd = str(row['home_cfbd'])
        away_cfbd = str(row['away_cfbd'])

        h = fuzz.token_sort_ratio(home_sched, home_cfbd)
        a = fuzz.token_sort_ratio(away_sched, away_cfbd)
        row['match_score'] = (h + a) / 2.0
        return row

    return candidates.apply(_calc, axis=1)


def greedy_match(
    candidates: pd.DataFrame,
    score_threshold: float,
    used_sched=None,
    used_cfbd=None
):
    """
    Greedy 1–1 matching:
      - sort by match_score desc
      - keep best available pairs
      - each sched_game_id and cfbd_game_id appears at most once
    """
    if used_sched is None:
        used_sched = set()
    if used_cfbd is None:
        used_cfbd = set()

    if candidates.empty:
        return pd.DataFrame(), used_sched, used_cfbd

    candidates = candidates.sort_values('match_score', ascending=False).copy()
    rows = []

    for r in candidates.itertuples(index=False):
        if r.match_score < score_threshold:
            continue
        if r.sched_game_id in used_sched or r.cfbd_game_id in used_cfbd:
            continue

        used_sched.add(r.sched_game_id)
        used_cfbd.add(r.cfbd_game_id)
        rows.append(r._asdict())

    if rows:
        matches_df = pd.DataFrame(rows)
    else:
        matches_df = pd.DataFrame(columns=candidates.columns)

    return matches_df, used_sched, used_cfbd


# -----------------------------------------------------------
# Main function: match + merge
# -----------------------------------------------------------

def match_cfbd_box(
    cfbd_raw: pd.DataFrame,
    box_raw: pd.DataFrame,
    time_window_att: float = 15,  # minutes for attendance-based candidates
    score_thr_att: float = 80,    # fuzzy threshold for attendance-based
    time_window_time: float = 10, # minutes for time-based candidates
    score_thr_time: float = 85    # fuzzy threshold for time-based
) -> pd.DataFrame:
    """
    Match cfbd and box dataframes and return a merged dataframe.

    ARGUMENTS (based on your setup):
      - cfbd_raw: dataframe with startDate, homeTeam, awayTeam, attendance, ...
      - box_raw:  dataframe with date, time_et, home, away, attendance, ...

    - Uses season, week, date, time, attendance, and fuzzy team names.
    - Team names in the final result default to the schedule (box_raw) home/away.
    - Only games that match are included.
    """

    # Prep
    sched = prepare_sched(box_raw)       # schedule: date + time_et + home/away
    cfbd = prepare_cfbd_box(cfbd_raw)    # cfbd: startDate + homeTeam/awayTeam

    # Minimal views for matching
    sched_small = sched[['sched_game_id', 'season', 'week', 'date_key', 'time_key',
                         'home', 'away', 'attendance']]
    cfbd_small = cfbd[['cfbd_game_id', 'season', 'week', 'date_key', 'time_key',
                       'homeTeam', 'awayTeam', 'attendance']]

    # ---------------------------------------------------
    # 1) Attendance-based candidates
    # ---------------------------------------------------
    sched_att = sched_small.dropna(subset=['attendance'])
    cfbd_att = cfbd_small.dropna(subset=['attendance'])

    cand_att = sched_att.merge(
        cfbd_att,
        on=['season', 'week', 'date_key', 'attendance'],
        suffixes=('_sched', '_cfbd')
    )

    if not cand_att.empty:
        cand_att = cand_att.rename(columns={
            'home': 'home_sched',
            'away': 'away_sched',
            'homeTeam': 'home_cfbd',
            'awayTeam': 'away_cfbd'
        })

        # Attach exact datetimes for time window filter
        cand_att = cand_att.merge(
            sched[['sched_game_id', 'game_dt_et']],
            on='sched_game_id'
        ).merge(
            cfbd[['cfbd_game_id', 'game_dt_et']],
            on='cfbd_game_id',
            suffixes=('_sched_dt', '_cfbd_dt')
        )

        cand_att['time_diff_min'] = (
            (cand_att['game_dt_et_sched_dt'] - cand_att['game_dt_et_cfbd_dt'])
            .abs()
            .dt.total_seconds() / 60.0
        )

        cand_att = cand_att[cand_att['time_diff_min'] <= time_window_att]

        if not cand_att.empty:
            cand_att = add_team_match_score(cand_att)
            matches_att, used_sched, used_cfbd = greedy_match(
                cand_att, score_thr_att
            )
        else:
            matches_att = pd.DataFrame()
            used_sched, used_cfbd = set(), set()
    else:
        matches_att = pd.DataFrame()
        used_sched, used_cfbd = set(), set()

    # ---------------------------------------------------
    # 2) Time-based candidates (no attendance)
    # ---------------------------------------------------
    sched_small_rem = sched_small[~sched_small['sched_game_id'].isin(used_sched)]
    cfbd_small_rem = cfbd_small[~cfbd_small['cfbd_game_id'].isin(used_cfbd)]

    cand_time = sched_small_rem.merge(
        cfbd_small_rem,
        on=['season', 'week', 'date_key', 'time_key'],
        suffixes=('_sched', '_cfbd')
    )

    if not cand_time.empty:
        cand_time = cand_time.rename(columns={
            'home': 'home_sched',
            'away': 'away_sched',
            'homeTeam': 'home_cfbd',
            'awayTeam': 'away_cfbd'
        })

        cand_time = cand_time.merge(
            sched[['sched_game_id', 'game_dt_et']],
            on='sched_game_id'
        ).merge(
            cfbd[['cfbd_game_id', 'game_dt_et']],
            on='cfbd_game_id',
            suffixes=('_sched_dt', '_cfbd_dt')
        )

        cand_time['time_diff_min'] = (
            (cand_time['game_dt_et_sched_dt'] - cand_time['game_dt_et_cfbd_dt'])
            .abs()
            .dt.total_seconds() / 60.0
        )

        cand_time = cand_time[cand_time['time_diff_min'] <= time_window_time]

        if not cand_time.empty:
            cand_time = add_team_match_score(cand_time)
            matches_time, used_sched, used_cfbd = greedy_match(
                cand_time, score_thr_time, used_sched, used_cfbd
            )
        else:
            matches_time = pd.DataFrame()
    else:
        matches_time = pd.DataFrame()

    # ---------------------------------------------------
    # 3) Combine matches and build final merged dataframe
    # ---------------------------------------------------
    matches = pd.concat([matches_att, matches_time], ignore_index=True)

    if matches.empty:
        # No matches found
        return pd.DataFrame()

    sched_full = sched.set_index('sched_game_id')
    cfbd_full = cfbd.set_index('cfbd_game_id')

    merged = matches.join(
        sched_full, on='sched_game_id', rsuffix='_sched_full'
    ).join(
        cfbd_full, on='cfbd_game_id', rsuffix='_cfbd_full'
    )

    # Drop cfbd team name columns if you want schedule names as canonical
    merged = merged.drop(columns=['homeTeam', 'awayTeam'], errors='ignore')

    # If you prefer canonical names to be 'homeTeam'/'awayTeam', use:
    # merged = merged.rename(columns={'home': 'homeTeam', 'away': 'awayTeam'})

    return merged

In [11]:
merged_games = match_cfbd_box(cfbd, box)
print(merged_games[['season', 'week', 'date_key', 'home', 'away']].head())

/var/folders/8l/f7sy6_5s2f76flj4cpkqxftr0000gn/T/ipykernel_4167/3909872013.py:24: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  sched['game_dt_et'] = pd.to_datetime(


   season  week    date_key        home            away
0    2002   1.0  2002-08-22    Virginia  Colorado State
1    2018   7.0  2018-10-13      Auburn       Tennessee
2    2018   6.0  2018-10-06  Ohio State         Indiana
3    2018   6.0  2018-10-06        UNLV      New Mexico
4    2018   6.0  2018-10-06    Colorado   Arizona State


In [13]:
len(merged_games)

16547

In [14]:
merged_games.to_csv("/Users/will/GitHub/IS477/data/cleaned/merged_games.csv")

In [19]:

def clean_team_name(name):
    if pd.isna(name):
        return None
    name = str(name).lower().strip()

    replacements = {
        "&": "and",
        "st.": "state",
        "st": "state ",
        "university": "",
        "univ": "",
        "u ": "",
        "-": " ",
        ".":""
    }

    for old, new in replacements.items():
        name = name.replace(old, new)
    
    return " ".join(name.split())

for df, cols in [(cfbd, ["homeTeam", "awayTeam"]), (box, ["home", "away"])]:
    df["home_clean"] = df[cols[0]].map(clean_team_name)
    df["away_clean"] = df[cols[0]].map(clean_team_name)


def parse_et_to_utc(dt_string):
    if pd.isna(dt_string):
        return None
    local = parser.parse(dt_string)
    eastern = pytz.timezone("US/Eastern")
    local = eastern.localize(local)
    return local.astimezone(pytz.utc)

cfbd["game_datetime_utc"] = pd.to_datetime(cfbd["startDate"], utc=True)
box["game_datetime_utc"] = box["date"].map(parse_et_to_utc)

for df in [cfbd, box]:
    df["game_datetime_round"] = df["game_datetime_utc"].dt.round("10min")

def fuzzy_match_name(name,choices, threshold=0.8):
    if pd.isna(name):
        return None
    matches = difflib.get_close_matches(name, choices, n=1, cutoff=threshold)
    return matches[0] if matches else None

cfbd_names = list(cfbd["home_clean"].dropna().unique())

box["home_clean_fuzzy"] = box["home_clean"].map(lambda x: fuzzy_match_name(x, cfbd_names))
box["away_clean_fuzzy"] = box["away_clean"].map(lambda x: fuzzy_match_name(x, cfbd_names))


merged = cfbd.merge(
    box,
    how="inner",
    left_on=["game_datetime_round", "home_clean", "away_clean"],
    right_on=["game_datetime_round", "home_clean_fuzzy", "away_clean_fuzzy"],
    suffixes=("_cfbd","_box")
)

print("Merged sample:")
display(merged.head())

print(f"Merged rows: {len(merged)}")

4